# Le Transformer, forme par forme

Ce notebook fait traverser le Transformer from scratch a un vrai batch, et affiche la forme des
tenseurs a chaque etape. Il suit la section 3 de [GUIDE.md](../GUIDE.md), dans le meme ordre.

**Les poids sont aleatoires.** Le modele est construit ici, il n'est pas entraine : ce notebook
montre la plomberie, pas la qualite. Les resumes produits a la derniere cellule sont donc du
charabia, et c'est normal. Pour un modele qui a appris, voir `02_training.ipynb`.

Rien n'est ecrit sur le disque. Le notebook tourne sur CPU en une minute environ.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.config import TokenizerConfig, load_pipeline_config
from src.data.dataset import MANIFEST_NAME, load_split
from src.data.example import Example
from src.data.tokenize import IGNORE_INDEX, build_tokenizer, encode_batch
from src.models.generation import GenerationConfig
from src.models.scratch.attention import scaled_dot_product_attention
from src.models.scratch.config import ScratchTransformerConfig
from src.models.scratch.generation import generate
from src.models.scratch.masks import build_decoder_mask, build_padding_mask
from src.models.scratch.transformer import ScratchTransformer, shift_target_right
from src.utils.seed import set_seed


def shape(label: str, tensor: torch.Tensor) -> None:
    """Print one tensor shape, aligned."""
    print(f"{label:<34} {tuple(tensor.shape)}")


set_seed(42)
print(f"racine du depot   {REPO_ROOT}")
print(f"torch             {torch.__version__}")

## 1. Deux exemples

On prend deux documents du split de test si le corpus est construit, deux documents ecrits en dur
sinon. Le reste du notebook ne fait pas la difference.

In [ ]:
FALLBACK = [
    Example(
        example_id="fallback-0",
        source=(
            "Three men have been arrested on suspicion of murder after a man was found "
            "badly injured at a meat market in the Old Swan area of Liverpool. The victim, "
            "aged 44, was shot in the head and chest and died later in hospital. The three "
            "men remain in police custody for questioning."
        ),
        target=(
            "Three men have been arrested on suspicion of murder over a shooting at a "
            "meat market."
        ),
    ),
    Example(
        example_id="fallback-1",
        source=(
            "The council has approved plans for a new footbridge over the river, replacing "
            "a crossing closed for safety reasons in 2019. Work is expected to start in the "
            "spring and to take eighteen months. Local traders welcomed the decision."
        ),
        target="Plans for a new footbridge over the river have been approved by the council.",
    ),
]

pipeline = load_pipeline_config(REPO_ROOT / "configs" / "data" / "xsum.yaml")
processed = Path(pipeline.paths.processed)
if not processed.is_absolute():
    processed = REPO_ROOT / processed

if (processed / MANIFEST_NAME).is_file():
    examples = load_split(processed, "test")[:2]
    print(f"corpus reel, {processed}")
else:
    examples = FALLBACK
    print("corpus absent, exemples de secours (make data pour le construire)")

for example in examples:
    print()
    print(f"  document : {example.source[:110]}...")
    print(f"  reference: {example.target}")

## 2. Tokenisation

Le tokenizer T5 transforme le texte en entiers. Les deux modeles du projet le partagent, donc
l'entree est strictement la meme des deux cotes.

Quatre tenseurs sortent de `encode_batch`. `target_ids` sert a construire l'entree du decodeur,
`labels` sert a calculer la loss : la difference est le `-100` pose sur le padding, valeur que
l'entropie croisee ignore.

In [ ]:
tokenizer_config = TokenizerConfig(
    hf_id=pipeline.tokenizer.hf_id,
    source_prefix=pipeline.tokenizer.source_prefix,
    max_source_tokens=pipeline.tokenizer.max_source_tokens,
    max_target_tokens=pipeline.tokenizer.max_target_tokens,
)
tokenizer = build_tokenizer(tokenizer_config.hf_id)
VOCAB_SIZE = len(tokenizer)

batch = encode_batch(examples, tokenizer, tokenizer_config)

print(f"vocabulaire       {VOCAB_SIZE}")
print(f"prefixe source    {tokenizer_config.source_prefix!r}")
print(f"plafond source    {tokenizer_config.max_source_tokens} tokens")
print(f"plafond cible     {tokenizer_config.max_target_tokens} tokens")
print()
shape("input_ids", batch.input_ids)
shape("attention_mask", batch.attention_mask)
shape("target_ids", batch.target_ids)
shape("labels", batch.labels)

print()
print("Les 12 premiers identifiants du document 0 :")
print(" ", batch.input_ids[0, :12].tolist())
print()
print("target_ids contre labels, fin de la sequence 1 :")
print("  target_ids", batch.target_ids[1, -6:].tolist())
print("  labels    ", batch.labels[1, -6:].tolist(), "  <- -100 la ou il y a du padding")
print()
print(
    f"positions comptees par la loss : {int((batch.labels != IGNORE_INDEX).sum())}"
    f" sur {batch.labels.numel()}"
)

## 3. Le modele

Configuration de `scratch_100` : `d_model=256`, 8 tetes donc `d_head=32`, 4 couches d'encodeur,
4 de decodeur, `d_ff=1024`.

Le compte de parametres se lit : plus de la moitie du modele est la seule table d'embedding.
C'est ce qui explique que l'ablation de profondeur ne montre rien, section 5 du rapport. Faire
varier le nombre de couches ne fait varier qu'une minorite des poids.

In [ ]:
config = ScratchTransformerConfig(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    num_heads=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    d_ff=1024,
    dropout=0.1,
    max_position=512,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=tokenizer.pad_token_id,
)
# eval() desactive le dropout : deux executions de ce notebook donnent le meme resultat.
model = ScratchTransformer(config).eval()

embedding_params = config.vocab_size * config.d_model
total = model.num_parameters
print(f"parametres        {total:,}".replace(",", " "))
print(
    f"dont embedding    {embedding_params:,}".replace(",", " ")
    + f"  ({embedding_params / total:.0%})"
)
print(f"d_head            {config.d_head}")

## 4. Embedding et encodage positionnel

L'embedding transforme `(batch, seq)` entiers en `(batch, seq, d_model)` flottants, puis multiplie
par `sqrt(d_model)`.

Ce facteur n'est pas cosmetique : sans lui, l'encodage positionnel qui vit dans `[-1, 1]`
ecraserait le signal de token. La cellule le mesure.

In [ ]:
with torch.no_grad():
    raw = model.embedding.embedding(batch.input_ids)   # table brute, sans le facteur
    embedded = model.embedding(batch.input_ids)        # avec le facteur sqrt(d_model)
    positioned = model.encoder.positional_encoding(embedded)

shape("input_ids", batch.input_ids)
shape("embedding brut", raw)
shape("embedding * sqrt(d_model)", embedded)
shape("+ encodage positionnel", positioned)

encoding = model.encoder.positional_encoding.get_buffer("encoding")
print()
print(f"amplitude embedding brut        {raw.abs().mean():.4f}")
print(f"amplitude apres mise a echelle  {embedded.abs().mean():.4f}")
print(f"amplitude encodage positionnel  {encoding.abs().mean():.4f}")
print()
print("Sans le facteur, le signal de position dominerait le signal de token.")

## 5. Q, K, V et le decoupage en tetes

Trois projections lineaires du meme tenseur, puis un decoupage en 8 tetes de 32 dimensions.

Le decoupage ne perd ni n'ajoute rien : `256 = 8 x 32`. C'est une operation de forme.

In [ ]:
attention_block = model.encoder.layers[0].self_attention

with torch.no_grad():
    query = attention_block.query_projection(positioned)
    key = attention_block.key_projection(positioned)
    value = attention_block.value_projection(positioned)

    heads_query = attention_block._split_heads(query)
    heads_key = attention_block._split_heads(key)
    heads_value = attention_block._split_heads(value)

shape("entree", positioned)
shape("Q = entree @ W_Q", query)
shape("K = entree @ W_K", key)
shape("V = entree @ W_V", value)
print()
shape("Q decoupe en tetes", heads_query)
shape("K decoupe en tetes", heads_key)
shape("V decoupe en tetes", heads_value)
print()
print(f"{config.d_model} = {config.num_heads} tetes x {config.d_head} dimensions")

## 6. Les scores d'attention et le masque

`Q Kt / sqrt(d_k)` donne un score par paire (requete, cle). C'est le seul tenseur dont la taille
croit comme le **carre** de la longueur : c'est lui qui rend la troncature a 512 tokens couteuse.

Le masque de padding interdit a toute requete de lire le remplissage. Apres le softmax, le poids
d'une position interdite est nul.

In [ ]:
source_mask = build_padding_mask(batch.input_ids, config.pad_token_id)

with torch.no_grad():
    d_k = heads_query.size(-1)
    scores = torch.matmul(heads_query, heads_key.transpose(-2, -1)) / (d_k**0.5)
    context, weights = scaled_dot_product_attention(
        heads_query, heads_key, heads_value, mask=source_mask
    )

shape("masque de padding", source_mask)
shape("scores Q Kt / sqrt(d_k)", scores)
shape("poids apres softmax", weights)
shape("contexte", context)

print()
print(f"somme des poids d'une requete : {float(weights[0, 0, 0].sum()):.4f}  (le softmax normalise)")

# Les documents de XSum atteignent souvent le plafond de 512, auquel cas le batch ne
# porte aucun padding. Deux phrases de longueurs franchement differentes le rendent
# visible a coup sur.
pair = [
    Example(example_id="court", source="A short sentence.", target="Short."),
    Example(
        example_id="long",
        source="A considerably longer sentence, written so that the batch has to be padded.",
        target="Longer.",
    ),
]
small = encode_batch(pair, tokenizer, tokenizer_config)
small_mask = build_padding_mask(small.input_ids, config.pad_token_id)

with torch.no_grad():
    embedded_small = model.encoder.positional_encoding(model.embedding(small.input_ids))
    _, small_weights = model.encoder.layers[0].self_attention(
        embedded_small, embedded_small, embedded_small, small_mask, return_weights=True
    )

real = int(small.attention_mask[0].sum())
width = small.input_ids.size(1)
print()
print(f"batch de demonstration : sequence 0 a {real} vrais tokens sur {width} positions")
print(f"poids max donne au padding par la requete 0 : {float(small_weights[0, 0, 0, real:].max()):.2e}")
print(f"poids max donne aux vrais tokens           : {float(small_weights[0, 0, 0, :real].max()):.2e}")

## 7. La sortie multi-tetes

Les tetes sont recomposees, puis `W_O` melange ce qu'elles ont recolte separement. On revient a
la largeur d'entree : un bloc d'attention preserve la forme, ce qui est exactement ce qu'il faut
pour l'empiler.

In [ ]:
with torch.no_grad():
    merged = attention_block._merge_heads(context)
    attended, _ = attention_block(positioned, positioned, positioned, source_mask)

shape("contexte par tete", context)
shape("tetes recomposees", merged)
shape("apres W_O", attended)
print()
print(f"forme d'entree preservee : {tuple(positioned.shape) == tuple(attended.shape)}")

## 8. L'encodeur : la memoire

Quatre couches identiques, chacune une self-attention puis un feed-forward, toutes deux
residuelles. La sortie est la **memoire** : le document lu une fois, que le decodeur consultera a
chacune de ses positions.

In [ ]:
with torch.no_grad():
    memory, memory_mask, _ = model.encode(batch.input_ids)

shape("input_ids", batch.input_ids)
shape("memoire de l'encodeur", memory)
shape("masque de la memoire", memory_mask)
print()
print(f"couches d'encodeur : {len(model.encoder.layers)}")

## 9. Le decalage a droite

Le decodeur ne doit jamais recevoir le token qu'on lui demande de predire. `shift_target_right`
decale la cible d'une position et prefixe le token de depart.

La cellule affiche les deux sequences l'une sous l'autre : a chaque colonne, le decodeur voit le
token du bas et doit predire celui du haut.

In [ ]:
decoder_input = shift_target_right(batch.target_ids, config.decoder_start_token_id)

shape("target_ids (la cible)", batch.target_ids)
shape("entree du decodeur", decoder_input)


def render(ids: torch.Tensor, count: int = 7) -> str:
    """Render the first tokens of a sequence as an aligned row."""
    cells = [tokenizer.decode([int(i)]).strip() or "_" for i in ids[:count]]
    return " | ".join(f"{cell:<10}" for cell in cells)


print()
print(f"{'cible':<16}", render(batch.target_ids[0]))
print(f"{'entree decodeur':<16}", render(decoder_input[0]))
print()
print("Le decodeur predit la ligne du haut a partir de la ligne du bas.")
print(f"token de depart : {config.decoder_start_token_id} (le padding, comme dans T5)")

## 10. Le decodeur et les logits

Chaque couche du decodeur enchaine trois sous-blocs : self-attention masquee, cross-attention sur
la memoire, feed-forward.

La projection finale envoie chaque position sur une entree du vocabulaire par score. Ce sont des
**logits** : pas encore des probabilites.

In [ ]:
target_mask = build_decoder_mask(decoder_input, config.pad_token_id)

with torch.no_grad():
    logits, _ = model.decode(decoder_input, memory, memory_mask)
    probabilities = logits.softmax(dim=-1)

shape("masque du decodeur", target_mask)
shape("logits", logits)
shape("probabilites", probabilities)

print()
print("Le masque du decodeur, coin haut gauche (1 = autorise) :")
print(target_mask[0, 0, :5, :5].int().numpy())
print()
print("Le triangle inferieur est le masque causal : la position t voit 0..t, jamais plus loin.")
print("La colonne 0 et la ligne 0 sont eteintes parce que le token de depart EST le padding,")
print("et que le masque de padding s'applique aussi. La position 0 n'a donc rien a regarder :")
print("c'est le cas limite pour lequel masked_fill_value rend le plus petit reel fini du type")
print("plutot que moins l'infini, qui donnerait ici une division 0/0, donc des NaN.")

best = int(logits[0, 0].argmax())
print()
print(f"token le plus probable en position 0 : {best} = {tokenizer.decode([best])!r}")
print(
    f"sa probabilite : {float(probabilities[0, 0, best]):.5f}"
    f"  (poids aleatoires, donc de l'ordre de 1/{VOCAB_SIZE})"
)

## 11. La loss

Entropie croisee entre les logits et les labels, les deux aplatis en une prediction par ligne.
Les positions de padding portent `-100` et sont retirees de la moyenne.

Un modele non entraine repartit sa masse a peu pres uniformement sur le vocabulaire, donc sa loss
doit valoir environ `ln(32100)`. La cellule le verifie : c'est le controle de sante le plus simple
d'une implementation de Transformer, et il attrape la plupart des erreurs de masque.

In [ ]:
import math

with torch.no_grad():
    output = model(batch.input_ids, target_ids=batch.target_ids, labels=batch.labels)

print(f"loss mesuree      {float(output.loss):.4f}")
print(f"ln(vocabulaire)   {math.log(VOCAB_SIZE):.4f}")
print()
print("Les deux sont proches : le modele n'a rien appris, il tire au hasard quasi uniforme.")
print("Un ecart franc ici signalerait un bug de masque ou de decalage.")

## 12. La generation

A l'entrainement, toute la cible est donnee d'un coup. A la generation, rien n'est donne : le
modele produit un token, se le redonne, et recommence.

Les poids sont aleatoires, donc **la sortie est du charabia**. Ce que la cellule montre est que la
boucle autoregressive tourne et rend la bonne forme.

In [ ]:
with torch.no_grad():
    greedy = generate(model, batch.input_ids, GenerationConfig(max_new_tokens=12, num_beams=1))
    beams = generate(model, batch.input_ids, GenerationConfig(max_new_tokens=12, num_beams=4))

shape("greedy (num_beams=1)", greedy)
shape("beam search (num_beams=4)", beams)

print()
for index, sequence in enumerate(greedy):
    print(f"  greedy {index} : {tokenizer.decode(sequence, skip_special_tokens=True)!r}")
print()
print("Charabia attendu : le modele n'a pas ete entraine. La plomberie, elle, fonctionne.")

## Recapitulatif

```text
input_ids                (2, S)
  embedding              (2, S, 256)
  + position             (2, S, 256)
  Q, K, V                (2, S, 256)
  en tetes               (2, 8, S, 32)
  scores                 (2, 8, S, S)     <- quadratique en S
  contexte               (2, 8, S, 32)
  recompose              (2, S, 256)
memoire                  (2, S, 256)
decalage a droite        (2, T)
logits                   (2, T, 32100)
loss                     scalaire
```

## La suite

- `02_training.ipynb` entraine reellement une experience et trace ses courbes.
- [GUIDE.md](../GUIDE.md), section 4, decrit ce qui arrive au batch apres la loss :
  retropropagation, optimiseur, scheduler, validation, checkpoints.
- `tests/unit/test_scratch_attention.py` et `test_scratch_masks.py` verifient les proprietes
  montrees ici, et echoueraient si elles etaient fausses.